# Collect ~1000 Applied CS Papers from arXiv (cs.LG · cs.CV · cs.CL · cs.RO · cs.SE · cs.DB · cs.AR · cs.NI · cs.HC · cs.IR · cs.AI)

This notebook queries arXiv for applied CS categories, applies a lightweight applied-keyword filter plus a theory-exclusion filter, removes duplicates, and saves outputs to CSV and JSON.

Target categories: `cs.LG` (Machine Learning), `cs.CV` (Computer Vision and Pattern Recognition), `cs.CL` (Computation and Language), `cs.RO` (Robotics), `cs.SE` (Software Engineering), `cs.DB` (Databases), `cs.AR` (Hardware Architecture), `cs.NI` (Networking and Internet Architecture), `cs.HC` (Human-Computer Interaction), `cs.IR` (Information Retrieval), `cs.AI` (Artificial Intelligence).

Rate-limit rules followed: 100 results per request, 3-second delay between requests.

In [1]:
# %pip install requests pandas
import re
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import requests

BASE_URL = "http://export.arxiv.org/api/query"
CATEGORIES = ["cs.LG", "cs.CV", "cs.CL", "cs.RO", "cs.SE", "cs.DB", "cs.AR", "cs.NI", "cs.HC", "cs.IR", "cs.AI"]
SEARCH_QUERY = " OR ".join(f"cat:{cat}" for cat in CATEGORIES)

TARGET_PAPERS = 1000
MAX_RESULTS_PER_CALL = 100
MAX_REQUESTS = 50
REQUEST_DELAY_SECONDS = 3
REQUEST_TIMEOUT_SECONDS = 60
MIN_KEYWORD_HITS = 2

APPLIED_KEYWORDS = [
    "experiment",
    "experiments",
    "experimental",
    "benchmark",
    "benchmarks",
    "benchmarking",
    "dataset",
    "datasets",
    "evaluation",
    "evaluate",
    "evaluating",
    "empirical",
    "empirically",
    "ablation",
    "ablation study",
    "implementation",
    "implement",
    "implemented",
    "we propose",
    "we introduce",
    "we present",
    "we build",
    "we design",
    "we develop",
    "we train",
    "we fine-tune",
    "we deploy",
    "performance",
    "accuracy",
    "throughput",
    "real-world",
    "real world",
    "state-of-the-art",
    "sota",
    "outperforms",
    "outperform",
    "user study",
    "human evaluation",
    "baseline",
    "baselines",
    "training",
    "inference",
    "results show",
    "we demonstrate",
    "we evaluate",
    "our method",
    "our model",
    "our system",
    "our approach",
    "our framework",
]

THEORY_EXCLUSION_KEYWORDS = [
    "theorem",
    "lemma",
    "corollary",
    "we prove",
    "formal proof",
    "np-hard",
    "np-complete",
    "lower bound",
    "upper bound",
    "convergence proof",
    "complexity analysis",
]

OUTPUT_DIR = Path("Dataset collection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "applied_papers_1000.csv"
OUTPUT_JSON = OUTPUT_DIR / "applied_papers_1000.json"

ATOM_NS = "{http://www.w3.org/2005/Atom}"
ARXIV_NS = "{http://arxiv.org/schemas/atom}"

session = requests.Session()
session.headers.update({
    "User-Agent": "ResearchPaperCategorizer/1.0 (mailto:replace-with-your-email@example.com)"
})

In [2]:
def clean_text(value: str) -> str:
    return " ".join((value or "").split())


APPLIED_PATTERNS = [re.compile(rf"\b{re.escape(keyword)}\b") for keyword in APPLIED_KEYWORDS]
THEORY_EXCLUSION_PATTERNS = [re.compile(rf"\b{re.escape(keyword)}\b") for keyword in THEORY_EXCLUSION_KEYWORDS]


def keyword_hits(text: str) -> list[str]:
    lowered = text.lower()
    return [keyword for keyword, pattern in zip(APPLIED_KEYWORDS, APPLIED_PATTERNS) if pattern.search(lowered)]


def theory_exclusion_hits(text: str) -> list[str]:
    lowered = text.lower()
    return [keyword for keyword, pattern in zip(THEORY_EXCLUSION_KEYWORDS, THEORY_EXCLUSION_PATTERNS) if pattern.search(lowered)]


def parse_entry(entry) -> dict:
    title = clean_text(entry.findtext(f"{ATOM_NS}title", default=""))
    abstract = clean_text(entry.findtext(f"{ATOM_NS}summary", default=""))
    arxiv_url = clean_text(entry.findtext(f"{ATOM_NS}id", default=""))
    published = clean_text(entry.findtext(f"{ATOM_NS}published", default=""))

    cat_terms = []
    for cat in entry.findall(f"{ATOM_NS}category"):
        term = clean_text(cat.attrib.get("term", ""))
        if term:
            cat_terms.append(term)

    primary_node = entry.find(f"{ARXIV_NS}primary_category")
    primary_category = ""
    if primary_node is not None:
        primary_category = clean_text(primary_node.attrib.get("term", ""))
    if not primary_category and cat_terms:
        primary_category = cat_terms[0]

    arxiv_id = arxiv_url.rsplit("/", 1)[-1] if arxiv_url else ""

    return {
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "published": published,
        "primary_category": primary_category,
        "categories": " ".join(sorted(set(cat_terms))),
        "arxiv_url": arxiv_url,
    }


def fetch_papers(start: int, max_results: int = 100) -> list[dict]:
    params = {
        "search_query": SEARCH_QUERY,
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    response = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    entries = root.findall(f"{ATOM_NS}entry")
    return [parse_entry(entry) for entry in entries]


def is_applied_like(paper: dict, min_hits: int = 2) -> bool:
    text = f"{paper['title']} {paper['abstract']}"
    hits = keyword_hits(text)
    paper["keyword_hits"] = ", ".join(hits)
    paper["keyword_hit_count"] = len(hits)
    return len(hits) >= min_hits


def passes_theory_exclusion(paper: dict, min_hits: int = 3) -> bool:
    hits = theory_exclusion_hits(paper["abstract"])
    paper["theory_exclusion_hits"] = ", ".join(hits)
    paper["theory_exclusion_hit_count"] = len(hits)
    return len(hits) < min_hits

In [3]:
applied_papers = []
seen_ids = set()
seen_titles = set()
rejected_by_theory_exclusion = 0

start = 0
for request_idx in range(1, MAX_REQUESTS + 1):
    batch = fetch_papers(start=start, max_results=MAX_RESULTS_PER_CALL)
    if not batch:
        print("No more results returned by arXiv.")
        break

    accepted = 0
    for paper in batch:
        if not paper["title"] or not paper["abstract"]:
            continue

        title_key = re.sub(r"\s+", " ", paper["title"].lower()).strip()
        if paper["arxiv_id"] in seen_ids or title_key in seen_titles:
            continue

        if is_applied_like(paper, min_hits=MIN_KEYWORD_HITS):
            if passes_theory_exclusion(paper):
                seen_ids.add(paper["arxiv_id"])
                seen_titles.add(title_key)
                applied_papers.append(paper)
                accepted += 1
            else:
                rejected_by_theory_exclusion += 1

        if len(applied_papers) >= TARGET_PAPERS:
            break

    print(
        f"Request {request_idx}/{MAX_REQUESTS} | start={start} | fetched={len(batch)} | accepted={accepted} | total={len(applied_papers)} | theory_rejected={rejected_by_theory_exclusion}"
    )

    if len(applied_papers) >= TARGET_PAPERS:
        break

    start += MAX_RESULTS_PER_CALL
    time.sleep(REQUEST_DELAY_SECONDS)

print(f"Collected {len(applied_papers)} applied-like papers.")
print(f"Rejected by theory exclusion filter: {rejected_by_theory_exclusion}")

Request 1/50 | start=0 | fetched=100 | accepted=88 | total=88 | theory_rejected=0
Request 2/50 | start=100 | fetched=100 | accepted=84 | total=172 | theory_rejected=0
Request 3/50 | start=200 | fetched=100 | accepted=88 | total=260 | theory_rejected=0
Request 4/50 | start=300 | fetched=100 | accepted=92 | total=352 | theory_rejected=0
Request 5/50 | start=400 | fetched=100 | accepted=92 | total=444 | theory_rejected=0
Request 6/50 | start=500 | fetched=100 | accepted=88 | total=532 | theory_rejected=0
Request 7/50 | start=600 | fetched=100 | accepted=89 | total=621 | theory_rejected=0
Request 8/50 | start=700 | fetched=100 | accepted=85 | total=706 | theory_rejected=0
Request 9/50 | start=800 | fetched=100 | accepted=93 | total=799 | theory_rejected=0
Request 10/50 | start=900 | fetched=100 | accepted=89 | total=888 | theory_rejected=0
Request 11/50 | start=1000 | fetched=100 | accepted=87 | total=975 | theory_rejected=0
Request 12/50 | start=1100 | fetched=100 | accepted=25 | total=10

In [4]:
df = pd.DataFrame(applied_papers)
if df.empty:
    raise RuntimeError("No papers collected. Try reducing MIN_KEYWORD_HITS or increasing MAX_REQUESTS.")

df = df.head(TARGET_PAPERS).copy()
df.insert(0, "label", "APPLIED")
df.insert(1, "source_query", SEARCH_QUERY)

# Keep a project-friendly column order.
ordered_cols = [
    "label",
    "title",
    "abstract",
    "primary_category",
    "categories",
    "keyword_hit_count",
    "keyword_hits",
    "published",
    "arxiv_id",
    "arxiv_url",
    "source_query",
]
df = df[ordered_cols]

df.to_csv(OUTPUT_CSV, index=False)
df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print(f"Saved {len(df):,} rows to {OUTPUT_CSV}")
print(f"Saved {len(df):,} rows to {OUTPUT_JSON}")
print()
print("Top primary categories:")
print(df["primary_category"].value_counts().head(10))
print()
print("Keyword hit distribution:")
print(df["keyword_hit_count"].value_counts().sort_index())

df.head(5)

Saved 1,000 rows to Dataset collection/applied_papers_1000.csv
Saved 1,000 rows to Dataset collection/applied_papers_1000.json

Top primary categories:
primary_category
cs.CV      322
cs.LG      195
cs.RO      102
cs.CL       98
cs.AI       76
cs.SE       23
cs.HC       18
cs.DB       16
cs.CR       16
stat.ML     16
Name: count, dtype: int64

Keyword hit distribution:
keyword_hit_count
2     112
3     129
4     182
5     183
6     144
7     104
8      75
9      34
10     24
11      6
12      4
13      2
14      1
Name: count, dtype: int64


,label,title,abstract,primary_category,categories,keyword_hit_count,keyword_hits,published,arxiv_id,arxiv_url,source_query
0,APPLIED,Towards Generalizable Robotic Manipulation in ...,Vision-Language-Action (VLA) models excel in s...,cs.CV,cs.CV cs.RO,12,"experiments, benchmark, dataset, datasets, eva...",2026-03-16T17:59:57Z,2603.15620v1,http://arxiv.org/abs/2603.15620v1,cat:cs.LG OR cat:cs.CV OR cat:cs.CL OR cat:cs....
1,APPLIED,Mixture-of-Depths Attention,Scaling depth is a key driver for large langua...,cs.CL,cs.AI cs.CL,6,"experiments, benchmarks, we introduce, perform...",2026-03-16T17:59:55Z,2603.15619v1,http://arxiv.org/abs/2603.15619v1,cat:cs.LG OR cat:cs.CV OR cat:cs.CL OR cat:cs....
2,APPLIED,Look Before Acting: Enhancing Vision Foundatio...,Vision-Language-Action (VLA) models have recen...,cs.CV,cs.CV,5,"we propose, we introduce, real-world, state-of...",2026-03-16T17:59:54Z,2603.15618v1,http://arxiv.org/abs/2603.15618v1,cat:cs.LG OR cat:cs.CV OR cat:cs.CL OR cat:cs....
3,APPLIED,HorizonMath: Measuring AI Progress Toward Math...,"Can AI make progress on important, unsolved ma...",cs.LG,cs.LG,5,"benchmark, benchmarks, evaluation, we introduc...",2026-03-16T17:59:53Z,2603.15617v1,http://arxiv.org/abs/2603.15617v1,cat:cs.LG OR cat:cs.CV OR cat:cs.CL OR cat:cs....
4,APPLIED,GlyphPrinter: Region-Grouped Direct Preference...,Generating accurate glyphs for visual text ren...,cs.CV,cs.CV,8,"experiments, dataset, we propose, we introduce...",2026-03-16T17:59:31Z,2603.15616v1,http://arxiv.org/abs/2603.15616v1,cat:cs.LG OR cat:cs.CV OR cat:cs.CL OR cat:cs....


In [5]:
# Quick manual spot-check (10 examples)
preview_cols = ["title", "primary_category", "keyword_hits"]
df.sample(n=min(10, len(df)), random_state=42)[preview_cols].reset_index(drop=True)

,title,primary_category,keyword_hits
0,Extending Minimal Pairs with Ordinal Surprisal...,cs.CL,"evaluation, evaluating"
1,Scene Generation at Absolute Scale: Utilizing ...,cs.CV,"we present, user study, baselines, our approach"
2,LineMaster Pro: A Low-Cost Intelligent Line Fo...,cs.RO,"experimental, benchmark, implementation, imple..."
3,Concisely Explaining the Doubt: Minimum-Size A...,cs.LG,"experimental, results show"
4,AtlasRAN: Modeling and Performance Evaluation ...,cs.NI,"experimental, evaluation, evaluating, performa..."
5,Amortizing Trajectory Diffusion with Keyed Dri...,cs.RO,"benchmarks, dataset, we introduce, performance..."
6,Towards Equitable Robotic Furnishing Agents fo...,cs.RO,"evaluating, we propose"
7,Histo-MExNet: A Unified Framework for Real-Wor...,cs.CV,"dataset, we propose, accuracy, real-world, tra..."
8,Unsupervised Adaptation from FDG to PSMA PET/C...,eess.IV,"we propose, we introduce, baseline, training"
9,Algorithms for Deciding the Safety of States i...,cs.AI,"experiments, benchmarks, performance"


In [6]:
from collections import Counter

category_counter = Counter()
for categories in df["categories"]:
    if not categories:
        continue
    category_counter.update(category for category in categories.split() if category)

applied_keyword_counter = Counter()
for abstract in df["abstract"]:
    applied_keyword_counter.update(keyword_hits(abstract))

print(f"Total papers collected: {len(df):,}")
print()
print("Distribution of papers per arXiv category:")
for category, count in category_counter.most_common():
    print(f"{category}: {count}")
print()
print("Top 10 most frequent applied keywords found across all abstracts:")
for keyword, count in applied_keyword_counter.most_common(10):
    print(f"{keyword}: {count}")
print()
print(f"Papers rejected by theory exclusion filter: {rejected_by_theory_exclusion:,}")

Total papers collected: 1,000

Distribution of papers per arXiv category:
cs.CV: 372
cs.AI: 331
cs.LG: 330
cs.CL: 138
cs.RO: 133
stat.ML: 34
cs.HC: 30
cs.SE: 30
cs.IR: 27
cs.CR: 27
cs.DB: 25
cs.SD: 21
cs.AR: 16
cs.NI: 16
cs.DC: 15
eess.IV: 15
cs.CY: 13
cs.MA: 13
cs.MM: 11
eess.SY: 11
math.OC: 9
cs.NE: 9
eess.AS: 8
cond-mat.mtrl-sci: 7
math.NA: 6
cs.LO: 5
cs.IT: 5
eess.SP: 5
quant-ph: 5
physics.comp-ph: 4
cs.GR: 4
stat.ME: 4
math.DS: 4
cs.ET: 4
q-bio.QM: 4
q-bio.NC: 4
stat.AP: 3
physics.chem-ph: 3
math.ST: 3
cs.DS: 3
cs.CE: 3
physics.soc-ph: 2
math.AG: 2
cond-mat.dis-nn: 2
cs.GT: 2
cs.PL: 2
cs.PF: 2
physics.app-ph: 1
physics.optics: 1
astro-ph.GA: 1
astro-ph.IM: 1
math.AP: 1
cs.DL: 1
physics.ao-ph: 1
q-bio.PE: 1
nlin.CD: 1
math.AT: 1
math.PR: 1
cond-mat.mes-hall: 1
q-bio.BM: 1
physics.ed-ph: 1
cond-mat.stat-mech: 1
astro-ph.CO: 1
stat.CO: 1
hep-ex: 1
cs.OS: 1
cs.SC: 1
physics.med-ph: 1
cs.SI: 1
cs.CC: 1
cs.FL: 1

Top 10 most frequent applied keywords found across all abstracts:
performa